# Project Aura 2.0 - Notebook 03: Model TrainingThis notebook trains a computer vision model using transfer learning:- Creates/loads a vision model from `src.vision.models`- Runs a training loop on preprocessed data- Saves checkpoints to local disk or GCS- Tracks training metricsImports reusable modules from `src/` where practical.

In [ ]:
import osimport sysimport timeimport numpy as npPROJECT_DIR = "/content/project-aura"if PROJECT_DIR not in sys.path:    sys.path.insert(0, PROJECT_DIR)!pip install --quiet torch torchvisionfrom src.core.config import load_configfrom src.vision.models import create_model, VisionModelBasefrom src.vision.preprocessing import resize_image, normalize_imagefrom src.utils.gcs import GCSClientfrom src.utils.logging import get_loggerlogger = get_logger("notebook-03")config = load_config()print("Configuration loaded")

In [ ]:
from PIL import Imageimport globDATA_DIR = "/content/project-aura/data"PROCESSED_DIR = os.path.join(DATA_DIR, "processed")CHECKPOINT_DIR = "/content/project-aura/checkpoints"os.makedirs(CHECKPOINT_DIR, exist_ok=True)if os.path.exists(PROCESSED_DIR):    image_files = sorted(glob.glob(os.path.join(PROCESSED_DIR, "*.jpg")))    print(f"Found {len(image_files)} processed images")else:    print("No processed images found. Generating synthetic data...")    os.makedirs(PROCESSED_DIR, exist_ok=True)    image_files = []    for i in range(20):        img = Image.new("RGB", (200, 200), (            np.random.randint(0, 255),            np.random.randint(0, 255),            np.random.randint(0, 255),        ))        path = os.path.join(PROCESSED_DIR, f"synthetic_{i:03d}.jpg")        img.save(path)        image_files.append(path)    print(f"Generated {len(image_files)} synthetic images")

In [ ]:
model = create_model(    model_name=config.vision.model_name,    num_classes=config.vision.num_classes,    pretrained=config.vision.pretrained,)print(f"Model: {model.model_name}")print(f"Variant: {getattr(model, "variant", "N/A")}")print(f"Classes: {model.num_classes}")print(f"Pretrained: {model.pretrained}")print(f"Device: {model._device}")

In [ ]:
import torchimport torch.nn as nnimport torch.optim as optimfrom torch.utils.data import DataLoader, TensorDatasetmodel.load()X_list = []y_list = []for i, img_path in enumerate(image_files):    img = np.array(Image.open(img_path).convert("RGB"))    resized = resize_image(img, target_size=config.vision.input_size)    normalized = normalize_image(resized)    chw = np.transpose(normalized, (2, 0, 1))    X_list.append(chw)    y_list.append(i % config.vision.num_classes)X = torch.from_numpy(np.stack(X_list).astype(np.float32))y = torch.from_numpy(np.array(y_list, dtype=np.int64))dataset = TensorDataset(X, y)dataloader = DataLoader(dataset, batch_size=config.vision.batch_size, shuffle=True)print(f"Training samples: {len(dataset)}")print(f"Batches per epoch: {len(dataloader)}")

In [ ]:
criterion = nn.CrossEntropyLoss()optimizer = optim.Adam(model._model.parameters(), lr=config.vision.learning_rate)num_epochs = min(config.vision.epochs, 3)device = model._devicemodel._model.to(device)model._model.train()training_history = []for epoch in range(num_epochs):    epoch_loss = 0.0    correct = 0    total = 0        for batch_X, batch_y in dataloader:        batch_X = batch_X.to(device)        batch_y = batch_y.to(device)                optimizer.zero_grad()        outputs = model._model(batch_X)        loss = criterion(outputs, batch_y)        loss.backward()        optimizer.step()                epoch_loss += loss.item()        _, predicted = torch.max(outputs.data, 1)        total += batch_y.size(0)        correct += (predicted == batch_y).sum().item()        avg_loss = epoch_loss / len(dataloader)    accuracy = 100 * correct / total    training_history.append({"epoch": epoch + 1, "loss": avg_loss, "accuracy": accuracy})    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f} Accuracy: {accuracy:.2f}%")print("\nTraining complete!")

In [ ]:
checkpoint_path = os.path.join(CHECKPOINT_DIR, f"{config.vision.model_name}_epoch{num_epochs}.pth")torch.save({    "model_name": config.vision.model_name,    "num_classes": config.vision.num_classes,    "epoch": num_epochs,    "model_state_dict": model._model.state_dict(),    "optimizer_state_dict": optimizer.state_dict(),    "training_history": training_history,}, checkpoint_path)print(f"Checkpoint saved to: {checkpoint_path}")gcs = GCSClient(    bucket_name=os.getenv("GCS_BUCKET_NAME"),    project_id=os.getenv("GCP_PROJECT_ID"),)if gcs._ensure_client():    blob_name = f"checkpoints/{os.path.basename(checkpoint_path)}"    success = gcs.upload_file(checkpoint_path, blob_name)    if success:        print(f"Checkpoint uploaded to GCS: {blob_name}")    else:        print("GCS upload failed")else:    print("GCS not available - checkpoint saved locally only")

In [ ]:
print("\n" + "="*60)print("Notebook 03 Complete - Model Training")print("="*60)print(f"Model: {config.vision.model_name}")print(f"Epochs: {num_epochs}")print(f"Final loss: {training_history[-1]["loss"]:.4f}")print(f"Final accuracy: {training_history[-1]["accuracy"]:.2f}%")print(f"Checkpoint: {checkpoint_path}")print("\nNext: Run 04_inference_engine.ipynb to test the model")